# 03 — Utiliser l'historique

**Question :** connaître les semaines passées d'un PDQ prédit-il mieux que le calendrier seul (boosting du notebook 02, MAE 5,17) ?
Et un modèle appris fait-il mieux qu'une règle simple comme « la moyenne des dernières semaines » ?

## Contrat de prédiction (v2)
**Pour chaque PDQ actif, chaque dimanche soir, estimer le nombre d'actes déclarés de la semaine suivante,
à partir du calendrier ET des semaines passées déjà consolidées.**

| Élément | v1 (notebooks 01–02) | v2 (ce notebook) |
|---|---|---|
| Unité, cible, moment | PDQ × semaine, actes déclarés, dimanche soir | identiques |
| Entrées | PDQ, mois, temps | PDQ, mois + **semaines passées consolidées** |

**Latence :** le dimanche soir, seules les données d'il y a plus de 14 jours sont sûres.
La semaine s−1 est presque vide et la semaine s−2 incomplète : **la plus récente semaine utilisable est s−3.**
Utiliser s−1 ou s−2 serait une fuite d'information : le modèle apprendrait avec des chiffres qu'il n'aura jamais au moment de prédire.

In [1]:
import json
import math
import hashlib
import tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
with open(ROOT / "configs/crimes.toml", "rb") as f:
    cfg = tomllib.load(f)

SEED = cfg["experiment"]["seed"]
DATA_PATH = ROOT / cfg["data"]["path"]
IN = ROOT / "artifacts" / "00_start_here"

panel = pd.read_parquet(IN / "panel.parquet")
meta = json.loads((IN / "meta.json").read_text())
meta_02 = json.loads((ROOT / "artifacts" / "02_arbres" / "meta.json").read_text())

empreinte = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()[:16]
assert empreinte == meta["source_sha16"], "Le CSV a changé depuis le notebook 00 : relance le 00 d'abord."
assert meta_02["source_sha16"] == meta["source_sha16"], "Le notebook 02 a été fait sur d'autres données : relance le 02."

panel["mois"] = panel["SEMAINE"].dt.month
print("Panel :", panel.shape)
print("À battre (notebook 02) :", meta_02["modele_retenu"], "| MAE validation", meta_02["mae_validation"])

Panel : (17835, 5)
À battre (notebook 02) : boosting | MAE validation 5.17


## Choix fixés avant d'entraîner
Le plus petit lag autorisé est **calculé** à partir de la latence de la config, puis vérifié.

In [2]:
H = cfg["historique"]
LAG_MIN = math.ceil(cfg["panel"]["lag_days"] / 7) + 1
LAGS = H["lags"]
FENETRES = H["fenetres"]

assert min(LAGS) >= LAG_MIN, f"Un lag de moins de {LAG_MIN} semaines utiliserait des semaines non consolidées."

print("Latence :", cfg["panel"]["lag_days"], "jours → plus petit lag autorisé :", LAG_MIN, "semaines")
print("Lags :", LAGS, "| Moyennes mobiles sur :", FENETRES, "semaines (à partir du lag", LAG_MIN, ")")

Latence : 14 jours → plus petit lag autorisé : 3 semaines
Lags : [3, 4, 52] | Moyennes mobiles sur : [4, 12] semaines (à partir du lag 3 )


## Construire l'historique de chaque PDQ
`lag_k` = actes du même PDQ il y a k semaines. `moy_w` = moyenne des w semaines consolidées les plus récentes (de s−3 à s−(w+2)).

In [3]:
panel = panel.sort_values(["PDQ", "SEMAINE"]).reset_index(drop=True)
par_pdq = panel.groupby("PDQ")["y"]

for k in LAGS:
    panel[f"lag_{k}"] = par_pdq.shift(k)

for w in FENETRES:
    panel[f"moy_{w}"] = par_pdq.transform(lambda s: s.shift(LAG_MIN).rolling(w).mean())

HISTOIRE = [f"lag_{k}" for k in LAGS] + [f"moy_{w}" for w in FENETRES]
panel.loc[panel["PDQ"] == "38", ["SEMAINE", "PDQ", "y"] + HISTOIRE].iloc[52:57]

,SEMAINE,PDQ,y,lag_3,lag_4,lag_52,moy_4,moy_12
9277,2016-01-04,38,41,45.0,65.0,36.0,51.75,51.416667
9278,2016-01-11,38,42,35.0,45.0,32.0,47.50,50.000000
9279,2016-01-18,38,28,25.0,35.0,40.0,42.50,47.666667
9280,2016-01-25,38,36,41.0,25.0,38.0,36.50,45.916667
9281,2016-02-01,38,29,42.0,41.0,36.0,35.75,45.166667


## Audit d'une ligne
On vérifie, pour le PDQ 38 et la semaine du 2 janvier 2023, que chaque lag vient de la bonne semaine
**et** que cette semaine était déjà consolidée au moment de prédire.

In [4]:
cible = pd.Timestamp("2023-01-02")
ligne = panel[(panel["PDQ"] == "38") & (panel["SEMAINE"] == cible)].iloc[0]
serie = panel[panel["PDQ"] == "38"].set_index("SEMAINE")["y"]

moment = cible - pd.Timedelta(days=1)
consolide = moment - pd.Timedelta(days=cfg["panel"]["lag_days"])
print("Semaine à prédire :", cible.date(), "| prédiction faite le", moment.date(), "| données sûres jusqu'au", consolide.date())

audit = []
for k in LAGS:
    source = cible - pd.Timedelta(weeks=k)
    fin_source = source + pd.Timedelta(days=6)
    assert ligne[f"lag_{k}"] == serie[source]
    assert fin_source <= consolide, "Cette semaine n'était pas encore consolidée !"
    audit.append({"colonne": f"lag_{k}", "semaine_source": source.date(), "fin_source": fin_source.date(),
                  "valeur": ligne[f"lag_{k}"]})
pd.DataFrame(audit)

Semaine à prédire : 2023-01-02 | prédiction faite le 2023-01-01 | données sûres jusqu'au 2022-12-18


,colonne,semaine_source,fin_source,valeur
0,lag_3,2022-12-12,2022-12-18,36.0
1,lag_4,2022-12-05,2022-12-11,39.0
2,lag_52,2022-01-03,2022-01-09,17.0


## Couverture : quelles semaines ont tout leur historique ?
`lag_52` demande un an de passé : les semaines de 2015 n'en ont pas.

In [5]:
complet = panel[HISTOIRE].notna().all(axis=1)
couverture = pd.DataFrame({
    "avant": panel.groupby("ENSEMBLE").size(),
    "avec_historique": panel[complet].groupby("ENSEMBLE").size(),
})
couverture["perdues"] = couverture["avant"] - couverture["avec_historique"]

panel = panel[complet].copy()
print("Première semaine utilisable :", panel["SEMAINE"].min().date())
couverture

Première semaine utilisable : 2016-01-04


,avant,avec_historique,perdues
ENSEMBLE,,,
test,4234,4234,0
train,12093,10585,1508
validation,1508,1508,0


## Outil d'évaluation
`noter` calcule les mesures à partir de prédictions déjà faites (pour les règles simples) ;
`evaluer` entraîne un modèle puis appelle `noter`.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

METRIQUE = "mae"
GROS_PDQ = meta_02["tranche_decision"]

train = panel[panel["ENSEMBLE"] == "train"]
validation = panel[panel["ENSEMBLE"] == "validation"]

modeles = {}
resultats = []

def noter(nom, pred_validation, pred_train):
    erreur = validation["y"] - pred_validation
    gros = validation["PDQ"].isin(GROS_PDQ)
    resultats.append({
        "modele": nom,
        "mae_validation": mean_absolute_error(validation["y"], pred_validation),
        "rmse_validation": root_mean_squared_error(validation["y"], pred_validation),
        "mae_train": mean_absolute_error(train["y"], pred_train),
        "p90_erreur": erreur.abs().quantile(0.9),
        "mae_gros_pdq": erreur[gros].abs().mean(),
        "biais_gros_pdq": erreur[gros].mean(),
    })
    return pd.DataFrame(resultats).set_index("modele").round(3)

def evaluer(nom, estimateur, numeriques=(), categorielles=()):
    colonnes = list(numeriques) + list(categorielles)
    etapes = []
    if numeriques:
        etapes.append(("num", StandardScaler(), list(numeriques)))
    if categorielles:
        etapes.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(categorielles)))
    modele = make_pipeline(ColumnTransformer(etapes), estimateur)
    modele.fit(train[colonnes], train["y"])
    modeles[nom] = modele
    return noter(nom, modele.predict(validation[colonnes]), modele.predict(train[colonnes]))

## Références historiques (sans apprentissage)
Quatre règles simples : « comme il y a 3 semaines », « comme l'an passé », « moyenne des 4 dernières semaines consolidées », « moyenne des 12 ».

In [7]:
for colonne in ["lag_3", "lag_52", "moy_4", "moy_12"]:
    tableau = noter(f"ref_{colonne}", validation[colonne], train[colonne])
tableau

,mae_validation,rmse_validation,mae_train,p90_erreur,mae_gros_pdq,biais_gros_pdq
modele,,,,,,
ref_lag_3,6.371,8.531,5.793,14.000,9.258,0.342
ref_lag_52,7.051,9.422,6.566,15.000,10.892,3.877
ref_moy_4,5.262,7.126,4.908,11.500,7.927,0.471
ref_moy_12,4.957,6.809,4.885,10.583,8.016,0.391
